# Prueba Técnica - Limpieza e Integración de Datos

En este notebook se realiza el proceso de exploración, limpieza, transformación e integración de los archivos 'transacciones.csv' y 'clientes.json', utilizando PySpark.

# Importación de librerías

In [1]:
import os
import sys
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    initcap,
    when,
    regexp_replace,
    to_date,
    coalesce,
    lit,
    length,
    translate,
    count,
    isnan,
    abs,
    expr,
    concat_ws
)

from pyspark.sql.types import (
    IntegerType,
    DoubleType,
    StringType,
    DateType
)

# Inicio Sesión PySpark

In [2]:
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Prueba")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .getOrCreate()
)

print("Python:", spark.sparkContext.pythonExec)
print("Spark:", spark.version)

Python: c:\Users\user\Documents\prueba_tecnica_analista_junior_iris\.venv\Scripts\python.exe
Spark: 3.5.6


# Carga de Data

In [3]:
clientes = (
    spark.read
    .option("multiline", True)
    .json("../data/clientes.json")
)

transacciones = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv("../data/transacciones.csv")
)

# Exploración de la Data

In [4]:
clientes.printSchema()
clientes.show(5, truncate=False)

transacciones.printSchema()
transacciones.show(5, truncate=False)

root
 |-- activo: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- contacto: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- telefono: string (nullable = true)
 |-- fecha_alta: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- numero_documento: string (nullable = true)
 |-- productos: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- segmento: string (nullable = true)
 |-- tipo_documento: string (nullable = true)

+------+---------+--------------------------------------------+-----------+----------+------------------+----------------+----------------------------------+--------+--------------+
|activo|ciudad   |contacto                                    |fecha_alta |id_cliente|nombre            |numero_documento|productos                         |segmento|tipo_documento|
+------+---------+--------------------------------------------+-----------+--------

In [5]:
transacciones.printSchema()
transacciones.show(5, truncate=False)

root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- tipo_producto: string (nullable = true)
 |-- monto: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tasa_interes: string (nullable = true)
 |-- plazo_dias: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- sucursal: string (nullable = true)

+--------------+----------+----------+-------------+-------------+-------+------------+----------+-----+--------+------------+
|id_transaccion|fecha     |id_cliente|tipo_producto|monto        |moneda |tasa_interes|plazo_dias|canal|estado  |sucursal    |
+--------------+----------+----------+-------------+-------------+-------+------------+----------+-----+--------+------------+
|500058        |2025-03-04|1013      |CTA_AHORROS  |1.236.028 COP|Pesos  |NULL        |0         |Web  |pend    |Barranquilla|
|500240        |2024-02-16|1007      |cd

In [6]:
transacciones.select("estado").distinct().show(100, False)
transacciones.select("canal").distinct().show(100, False)

+----------+
|estado    |
+----------+
|APROBADA  |
|pend      |
|Rechazada |
|rechazada |
|Pendiente |
|Aprobada  |
|aprobado  |
|En proceso|
|PENDIENTE |
+----------+

+-----------------+
|canal            |
+-----------------+
|APP              |
|oficina          |
|ATM              |
|Cajero           |
|app movil        |
|App              |
|OFICINA          |
|web              |
|cajero automatico|
|Web              |
|Oficina          |
+-----------------+



In [7]:
transacciones.select("moneda").distinct().show(100, False)
transacciones.select("sucursal").distinct().show(100, False)

+-------+
|moneda |
+-------+
|Dolares|
|usd    |
|COP    |
|pesos  |
|cop    |
|Pesos  |
|$      |
|USD    |
|NULL   |
+-------+

+------------+
|sucursal    |
+------------+
|cali        |
|Bucaramanga |
|b/quilla    |
|  Bogotá    |
|medellín    |
|Cali        |
|BOGOTA      |
|Medellin    |
|Bogota      |
|Cartagena   |
|Barranquilla|
|NULL        |
+------------+



In [8]:
transacciones.select("tipo_producto").distinct().show(100, False)
clientes.select("segmento").distinct().show(100, False)

+-----------------------+
|tipo_producto          |
+-----------------------+
|ahorros                |
|CDT                    |
|CORRIENTE              |
|CTA_AHORROS            |
|cdt                    |
|Crédito                |
|Credito                |
|crédito                |
|CERTIFICADO DE DEPOSITO|
|Certificado de Deposito|
|Cuenta Ahorros         |
|cta corriente          |
|Cuenta corriente       |
|AHORROS                |
|CREDITO                |
|Cdt                    |
|Cuenta Corriente       |
|Cuenta de ahorros      |
+-----------------------+

+-----------+
|segmento   |
+-----------+
|Premium    |
|Personal   |
|pyme       |
|Empresarial|
|Pyme       |
|EMPRESARIAL|
|personal   |
|premium    |
|NULL       |
+-----------+



In [9]:
clientes.select("tipo_documento").distinct().show(100, False)
clientes.select("activo").distinct().show(100, False)

+--------------+
|tipo_documento|
+--------------+
|cc            |
|nit           |
|NIT           |
|CC            |
|C.C.          |
|NULL          |
+--------------+

+------+
|activo|
+------+
|0     |
|No    |
|false |
|Si    |
|1     |
|SI    |
|NO    |
|true  |
+------+



# Exploración de Duplicados

In [10]:
transacciones.groupBy("id_transaccion") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+--------------+-----+
|id_transaccion|count|
+--------------+-----+
|        500148|    2|
|        500363|    2|
|        500153|    2|
|        500060|    2|
|        500123|    2|
|        500260|    2|
+--------------+-----+



In [11]:
transacciones.filter(
    col("id_transaccion").isin(
        "500148",
        "500363",
        "500153",
        "500060",
        "500123",
        "500260"
    )
).orderBy("id_transaccion").show(truncate=False)

+--------------+----------+----------+-----------------------+-------------+------+------------+----------+---------+----------+-----------+
|id_transaccion|fecha     |id_cliente|tipo_producto          |monto        |moneda|tasa_interes|plazo_dias|canal    |estado    |sucursal   |
+--------------+----------+----------+-----------------------+-------------+------+------------+----------+---------+----------+-----------+
|500060        |25.10.2024| 1026     |Cuenta de ahorros      |7.784.912,25 |NULL  |0           |0         |oficina  |En proceso|Bucaramanga|
|500060        |25.10.2024| 1026     |Cuenta de ahorros      |7.784.912,25 |NULL  |0           |0         |oficina  |En proceso|Bucaramanga|
|500123        |20-11-2024|7777      |ahorros                |1.973.695,25 |Pesos |N/A         |NULL      |app movil|Rechazada |b/quilla   |
|500123        |20-11-2024|7777      |ahorros                |1.973.695,25 |Pesos |N/A         |NULL      |app movil|Rechazada |b/quilla   |
|500148      

In [12]:
clientes.groupBy("id_cliente") \
    .count() \
    .filter(col("count") > 1) \
    .show(20, False)

+----------+-----+
|id_cliente|count|
+----------+-----+
|1004      |2    |
|1011      |2    |
+----------+-----+



In [13]:
clientes.filter(col("id_cliente") == "1004").show(20, False)

+------+--------+------------------------------------------+----------+----------+-------------+----------------+----------------+--------+--------------+
|activo|ciudad  |contacto                                  |fecha_alta|id_cliente|nombre       |numero_documento|productos       |segmento|tipo_documento|
+------+--------+------------------------------------------+----------+----------+-------------+----------------+----------------+--------+--------------+
|true  |medellín|{isabella.diaz@empresa.com.co, 3138849654}|7-1-2022  |1004      |Isabella Diaz|342601915       |[Cuenta Ahorros]|premium |C.C.          |
|true  |medellín|{isabella.diaz@empresa.com.co, 3138849654}|7-1-2022  |1004      |Isabella Diaz|342601915       |[Cuenta Ahorros]|premium |C.C.          |
+------+--------+------------------------------------------+----------+----------+-------------+----------------+----------------+--------+--------------+



In [14]:
clientes.filter(col("id_cliente") == "1011").show(20, False)

+------+------+----------------------+----------+----------+---------------+----------------+------------------+--------+--------------+
|activo|ciudad|contacto              |fecha_alta|id_cliente|nombre         |numero_documento|productos         |segmento|tipo_documento|
+------+------+----------------------+----------+----------+---------------+----------------+------------------+--------+--------------+
|false |cali  |{NULL, (602) 560 7689}|2022-11-27|1011      |SANTIAGO VARGAS|339147326       |[Cuenta Corriente]|premium |NIT           |
|false |cali  |{NULL, (602) 560 7689}|2022-11-27|1011      |SANTIAGO VARGAS|339147326       |[Cuenta Corriente]|premium |NIT           |
+------+------+----------------------+----------+----------+---------------+----------------+------------------+--------+--------------+



# Eliminación de Duplicados

In [15]:
print("Cantidad antes:", transacciones.count())
transacciones = transacciones.dropDuplicates()
print("Cantidad después:", transacciones.count())

Cantidad antes: 429
Cantidad después: 423


In [16]:
clientes = clientes.dropDuplicates()
clientes.groupBy("id_cliente") \
    .count() \
    .filter(col("count") > 1) \
    .show(20, False)


+----------+-----+
|id_cliente|count|
+----------+-----+
+----------+-----+



# Eliminar Espacios

In [17]:
for columna in transacciones.columns:
    transacciones = transacciones.withColumn(
        columna,
        trim(col(columna))
    )

# Exploración de Valores Vacios

In [18]:
for columna in transacciones.columns:
    cantidad = transacciones.filter(col(columna) == "").count()
    print(f"{columna}: {cantidad}")

id_transaccion: 0
fecha: 0
id_cliente: 0
tipo_producto: 0
monto: 0
moneda: 0
tasa_interes: 0
plazo_dias: 0
canal: 0
estado: 0
sucursal: 0


# Limpieza Transacciones

# Unificar texto en columna tipo_producto

In [19]:
transacciones = transacciones.replace(
    {
        "CERTIFICADO DE DEPOSITO": "CDT",
        "CTA_AHORROS": "CUENTA DE AHORROS",
        "AHORROS": "CUENTA DE AHORROS",
        "CUENTA AHORROS": "CUENTA DE AHORROS",
        "CTA CORRIENTE": "CUENTA CORRIENTE",
        "CORRIENTE": "CUENTA CORRIENTE",
        "CREDITO": "CRÉDITO"
    },
    subset=["tipo_producto"]
)

In [20]:
transacciones = transacciones.withColumn(
    "tipo_producto",
    initcap(col("tipo_producto"))
)

In [21]:
transacciones = transacciones.withColumn(
    "tipo_producto",
    regexp_replace(col("tipo_producto"), " De ", " de ")
)

In [22]:
transacciones.select("tipo_producto").distinct().show(truncate=False)

+-----------------------+
|tipo_producto          |
+-----------------------+
|Ahorros                |
|Crédito                |
|Credito                |
|Certificado de Deposito|
|Cuenta Ahorros         |
|Cuenta de Ahorros      |
|Cdt                    |
|Cuenta Corriente       |
|Cta Corriente          |
+-----------------------+



# Convertir Cdt --> CDT

In [23]:
transacciones = transacciones.withColumn(
    "tipo_producto",
    when(col("tipo_producto") == "Cdt", "CDT")
    .otherwise(col("tipo_producto"))
)

In [24]:
transacciones.select("tipo_producto").distinct().show(truncate=False)

+-----------------------+
|tipo_producto          |
+-----------------------+
|Ahorros                |
|CDT                    |
|Crédito                |
|Credito                |
|Certificado de Deposito|
|Cuenta Ahorros         |
|Cuenta de Ahorros      |
|Cuenta Corriente       |
|Cta Corriente          |
+-----------------------+



# Limpieza de la columna estado

# Unificar columna estado

In [25]:
transacciones = transacciones.withColumn(
    "estado",
    upper(col("estado"))
)

# Estandarizar columna estado

In [26]:
transacciones = transacciones.replace(
    {
        "APROBADO": "APROBADA",
        "PEND": "PENDIENTE",
        "EN PROCESO": "PENDIENTE"
    },
    subset=["estado"]
)

In [27]:
transacciones = transacciones.withColumn(
    "estado",
    initcap(col("estado"))
)

In [28]:
transacciones.select("estado").distinct().show()

+---------+
|   estado|
+---------+
|Rechazada|
|Pendiente|
| Aprobada|
+---------+



# Limpieza columna canal

# Unificar columna canal

In [29]:
transacciones = transacciones.withColumn(
    "canal",
    upper(col("canal"))
)

# Estandarizar columna canal

In [30]:
transacciones = transacciones.replace(
    {
        "APP MOVIL": "APP",
        "ATM": "CAJERO",
        "CAJERO AUTOMATICO": "CAJERO"
    },
    subset=["canal"]
)

In [31]:
transacciones = transacciones.withColumn(
    "canal",
    initcap(col("canal"))
)

In [32]:
transacciones.select("canal").distinct().show()

+-------+
|  canal|
+-------+
| Cajero|
|    App|
|    Web|
|Oficina|
+-------+



# Limpieza de la columna moneda

# Unificar columna moneda

In [33]:
transacciones = transacciones.withColumn(
    "moneda",
    upper(col("moneda"))
)

# Estandarizar columna moneda

In [34]:
transacciones = transacciones.replace(
    {
        "PESOS": "COP",
        "$": "COP",
        "DOLARES": "USD"
    },
    subset=["moneda"]
)

In [35]:
transacciones.select("moneda").distinct().show()

+------+
|moneda|
+------+
|   COP|
|   USD|
|  NULL|
+------+



# Limpieza de la columna monto

In [36]:
transacciones.select("monto").distinct().show(100, truncate=False)

+---------------+
|monto          |
+---------------+
|$6.711.031     |
|$7.043.062     |
|-10.805.060,25 |
|10.949.578     |
|44.129.606     |
|63.582.790     |
|$69.475.728    |
|63.275.519,25  |
|$98.426.847    |
|115.300.157 COP|
|76.033.808     |
|$18.577.287    |
|$5.544.806     |
|$88.284.618    |
|$4.841.651     |
|12.659.882 COP |
|12.304.083     |
|10.589.193,90  |
|sin dato       |
|2.388.293,00   |
|$10.363.721    |
|79.062.312,00  |
|$57.692.951    |
|67.675.881,00  |
|$4.921.055     |
|10.056.762,90  |
|$15.410.701    |
|94.688.983     |
|$14.836.914    |
|$6.709.024     |
|8.855.108 COP  |
|31.446.430     |
|5.934.639      |
|8.019.130 COP  |
|$12.568.254    |
|9.288.330 COP  |
|6,456.51       |
|$5.649.088     |
|-3.877.402     |
|2.841.745,25   |
|11.675.042,00  |
|8.349.971      |
|$966.918       |
|2.386.490,00   |
|$87.798.609    |
|$4.093.431     |
|64.596.891     |
|$31.075.165    |
|7.633.839 COP  |
|12.184.926     |
|24.286.585 COP |
|5,296.36       |
|$11.727.4

# Backup de monto
se crea backup porque es un dato sensible, permitiendo comparar los valores originales con los transformados y recuperar la información en caso de detectar errores durante la limpieza.

In [37]:
transacciones = transacciones.withColumn(
    "monto_original",
    col("monto")
)

# Estandarizarion campos sin valor fijo

In [38]:
transacciones = transacciones.replace(
    {
        "N/A": None,
        "NULL": None,
        "sin dato": None
    },
    subset=["monto"]
)

In [39]:
transacciones.filter(col("monto").isNull()).show()

+--------------+----------+----------+--------------------+-----+------+------------+----------+-------+---------+------------+--------------+
|id_transaccion|     fecha|id_cliente|       tipo_producto|monto|moneda|tasa_interes|plazo_dias|  canal|   estado|    sucursal|monto_original|
+--------------+----------+----------+--------------------+-----+------+------------+----------+-------+---------+------------+--------------+
|        500257|18/08/2024|      1049|    Cuenta Corriente| NULL|   COP|         N/A|         -|    Web| Aprobada|    b/quilla|           N/A|
|        500156|14-12-2023|      1048|      Cuenta Ahorros| NULL|   COP|         N/A|         0|Oficina| Aprobada|   Cartagena|      sin dato|
|        500265|22/02/2025|      1026|                 CDT| NULL|   COP|        8,7%|       180| Cajero|Pendiente|      BOGOTA|          NULL|
|        500064|2024-11-14|      1006|   Cuenta de Ahorros| NULL|   COP|         N/A|         -|    App|Pendiente|        cali|          NULL|

# Limpieza del simbolo $ y COP de la columna monto

In [40]:
transacciones = transacciones.withColumn(
    "monto",
    regexp_replace(col("monto"), "\\$", "")
)

In [41]:
transacciones = transacciones.withColumn(
    "monto",
    regexp_replace(col("monto"), " COP", "")
)

In [42]:
transacciones.select("monto").show(30, truncate=False)

+--------------+
|monto         |
+--------------+
|20,235.16     |
|76.033.808    |
|31.075.165    |
|58.964.224    |
|61.991.584    |
|-31.198.397,50|
|46.304.145    |
|26.641.248    |
|7.043.062     |
|94.085.540,00 |
|104.085.461,90|
|48.504.595    |
|26.629.119,00 |
|36.100.170    |
|59.872.735    |
|9.004.221     |
|8.864.973,50  |
|1.271.047     |
|62.221.217    |
|57.507.492    |
|14,233.99     |
|74.769.684    |
|66.630.171,90 |
|2.443.801     |
|8,559.32      |
|11.604.150    |
|23.191.663,50 |
|63.694.483    |
|49.096.276    |
|2.284.877,50  |
+--------------+
only showing top 30 rows



# Analizar moneda y monto
se analiza buscando relación entre su tipo de moneda y el separador de miles y decimales, en donde COP usa separador de miles el '.' y de decimales la ','

In [43]:
transacciones.select("moneda", "monto").distinct().show(100, truncate=False)

+------+--------------+
|moneda|monto         |
+------+--------------+
|COP   |59.059.045    |
|COP   |2.443.801     |
|COP   |8.019.130     |
|COP   |2.284.877,50  |
|COP   |12.659.882    |
|COP   |-1.070.297    |
|COP   |13.720.146    |
|USD   |16,193.57     |
|COP   |15.410.701    |
|COP   |6.709.024     |
|COP   |44.646.856    |
|USD   |7,241.11      |
|COP   |9.307.015,90  |
|COP   |6.073.220     |
|COP   |59.872.735    |
|COP   |72.967.304,90 |
|COP   |2.728.710,25  |
|COP   |94.688.983    |
|COP   |6.784.611     |
|COP   |5.357.374,25  |
|COP   |-10.805.060,25|
|USD   |16,451.98     |
|USD   |11,043.66     |
|COP   |3.421.231     |
|USD   |731.12        |
|COP   |11.097.696    |
|COP   |57.692.951    |
|COP   |45.815.428,00 |
|COP   |1.712.832,25  |
|COP   |13.623.446,50 |
|USD   |9,592.91      |
|COP   |35.101.509,90 |
|COP   |29.206.934    |
|USD   |24,610.95     |
|COP   |63.582.790    |
|COP   |2.447.012     |
|COP   |3.980.520     |
|COP   |12.735.704,25 |
|COP   |11.354.9

# Estandarización de separadores de miles y decimas en columna monto

In [44]:
transacciones = transacciones.withColumn(
    "monto",
    when(
        col("moneda") == "COP",
        regexp_replace(
            regexp_replace(col("monto"), "\\.", ""),
            ",",
            "."
        )
    ).when(
        col("moneda") == "USD",
        regexp_replace(col("monto"), ",", "")
    ).otherwise(col("monto"))
)

In [45]:
transacciones.select(
    "moneda",
    "monto_original",
    "monto"
).show(100, truncate=False)

+------+--------------+------------+
|moneda|monto_original|monto       |
+------+--------------+------------+
|USD   |20,235.16     |20235.16    |
|COP   |76.033.808    |76033808    |
|COP   |$31.075.165   |31075165    |
|COP   |$58.964.224   |58964224    |
|COP   |$61.991.584   |61991584    |
|COP   |-31.198.397,50|-31198397.50|
|COP   |46.304.145 COP|46304145    |
|COP   |26.641.248    |26641248    |
|COP   |$7.043.062    |7043062     |
|COP   |94.085.540,00 |94085540.00 |
|COP   |104.085.461,90|104085461.90|
|COP   |48.504.595    |48504595    |
|COP   |26.629.119,00 |26629119.00 |
|COP   |36.100.170    |36100170    |
|COP   |$59.872.735   |59872735    |
|COP   |9.004.221     |9004221     |
|COP   |8.864.973,50  |8864973.50  |
|COP   |$1.271.047    |1271047     |
|COP   |62.221.217    |62221217    |
|COP   |$57.507.492   |57507492    |
|USD   |14,233.99     |14233.99    |
|COP   |$74.769.684   |74769684    |
|COP   |66.630.171,90 |66630171.90 |
|COP   |$2.443.801    |2443801     |
|

In [46]:
transacciones.printSchema()

root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- tipo_producto: string (nullable = true)
 |-- monto: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tasa_interes: string (nullable = true)
 |-- plazo_dias: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- sucursal: string (nullable = true)
 |-- monto_original: string (nullable = true)



# Conversión de columna monto a tipo double

In [47]:
transacciones = transacciones.withColumn(
    "monto",
    col("monto").cast(DoubleType())
)

In [51]:
transacciones.printSchema()

root
 |-- id_transaccion: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- id_cliente: string (nullable = true)
 |-- tipo_producto: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tasa_interes: string (nullable = true)
 |-- plazo_dias: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- sucursal: string (nullable = true)
 |-- monto_original: string (nullable = true)
 |-- monto_cop: double (nullable = true)



In [48]:
transacciones.select(
    "monto_original",
    "monto"
).show(20, truncate=False)

+--------------+-------------+
|monto_original|monto        |
+--------------+-------------+
|20,235.16     |20235.16     |
|76.033.808    |7.6033808E7  |
|$31.075.165   |3.1075165E7  |
|$58.964.224   |5.8964224E7  |
|$61.991.584   |6.1991584E7  |
|-31.198.397,50|-3.11983975E7|
|46.304.145 COP|4.6304145E7  |
|26.641.248    |2.6641248E7  |
|$7.043.062    |7043062.0    |
|94.085.540,00 |9.408554E7   |
|104.085.461,90|1.040854619E8|
|48.504.595    |4.8504595E7  |
|26.629.119,00 |2.6629119E7  |
|36.100.170    |3.610017E7   |
|$59.872.735   |5.9872735E7  |
|9.004.221     |9004221.0    |
|8.864.973,50  |8864973.5    |
|$1.271.047    |1271047.0    |
|62.221.217    |6.2221217E7  |
|$57.507.492   |5.7507492E7  |
+--------------+-------------+
only showing top 20 rows



# Creación de nueva columna monto_cop

In [49]:
transacciones = transacciones.withColumn(
    "monto_cop",
    when(
        col("moneda") == "USD",
        col("monto") * 4000
    ).when(
        col("moneda") == "COP",
        col("monto")
    ).otherwise(None)
)

In [50]:
transacciones.select(
    "moneda",
    "monto",
    "monto_cop"
).show(30, truncate=False)

+------+-------------+-------------+
|moneda|monto        |monto_cop    |
+------+-------------+-------------+
|USD   |20235.16     |8.094064E7   |
|COP   |7.6033808E7  |7.6033808E7  |
|COP   |3.1075165E7  |3.1075165E7  |
|COP   |5.8964224E7  |5.8964224E7  |
|COP   |6.1991584E7  |6.1991584E7  |
|COP   |-3.11983975E7|-3.11983975E7|
|COP   |4.6304145E7  |4.6304145E7  |
|COP   |2.6641248E7  |2.6641248E7  |
|COP   |7043062.0    |7043062.0    |
|COP   |9.408554E7   |9.408554E7   |
|COP   |1.040854619E8|1.040854619E8|
|COP   |4.8504595E7  |4.8504595E7  |
|COP   |2.6629119E7  |2.6629119E7  |
|COP   |3.610017E7   |3.610017E7   |
|COP   |5.9872735E7  |5.9872735E7  |
|COP   |9004221.0    |9004221.0    |
|COP   |8864973.5    |8864973.5    |
|COP   |1271047.0    |1271047.0    |
|COP   |6.2221217E7  |6.2221217E7  |
|COP   |5.7507492E7  |5.7507492E7  |
|USD   |14233.99     |5.693596E7   |
|COP   |7.4769684E7  |7.4769684E7  |
|COP   |6.66301719E7 |6.66301719E7 |
|COP   |2443801.0    |2443801.0    |
|